# Stage 10 — Accuracy vs Ground Truth (LLM Query Expansion)

Evaluates **Stage 9** admissions using **LLM-based query expansion** for clinical synonym matching.

For each admission:

1. **DiffDx match** — does any ranked differential diagnosis match GT **primary diagnosis**? (any rank counts; `Is rank-1 match` column flags lower-rank hits)
2. **Linked ICD match** — does the ICD-10 code mapped to **that same matched DiffDx rank** match GT **primary ICD**?

No precision/recall/F1 code-set metrics.

**Output:**
- per admission: `accuracy/comparison.txt` + `accuracy/accuracy.json`
- cohort: `data/stage_10_evaluation/accuracy_summary.json` + `cohort_accuracy.txt`
- CSVs: `diffdx_match.csv`, `icd_match.csv`
- cache: `llm_qe_cache.json`

Requires Stage 8 `icd_coding.json` + Stage 9 `icd_coding_confirmed.json`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from pipeline import (
    EVAL_COHORT_TXT,
    EVAL_DIFFDX_CSV,
    EVAL_ICD_CSV,
    EVAL_SUMMARY_JSON,
    EXPORT_DIR,
    STAGE_10_DIR,
    print_pipeline_banner,
    run_stage10_evaluation,
)

print_pipeline_banner()
STAGE_10_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export dir : {EXPORT_DIR}")
print(f"Stage 10 out: {STAGE_10_DIR}")
print("Requires icd_coding.json + icd_coding_confirmed.json per admission.")

In [ ]:
payload = run_stage10_evaluation(export_dir=EXPORT_DIR, use_llm_qe=True)
summary = payload.get("summary") or {}
dx = summary.get("diagnosis") or {}
icd = summary.get("icd_linked") or {}
n = int(summary.get("n_scored") or 0)
print(f"\nAdmissions scored: {n}")
print(f"DiffDx match (any rank):  {dx.get('match_n', 0)}/{n}  ({100*float(dx.get('match_rate') or 0):.1f}%)")
print(f"DiffDx match (rank-1):    {dx.get('rank1_match_n', 0)}/{n}  ({100*float(dx.get('rank1_match_rate') or 0):.1f}%)")
print(f"Lower-rank only:          {dx.get('lower_rank_match_n', 0)}/{n}")
print(f"Linked ICD match:         {icd.get('match_n', 0)}/{n}  ({100*float(icd.get('match_rate') or 0):.1f}%)")
print(f"Linked ICD exact:         {icd.get('exact_match_n', 0)}/{n}")
print(f"Diagnosis YES, ICD NO:    {icd.get('diagnosis_yes_icd_no_n', 0)}")
print(f"\nSaved → {EVAL_SUMMARY_JSON}")
print(f"Cohort  → {EVAL_COHORT_TXT}")
print(f"DiffDx  → {EVAL_DIFFDX_CSV}")
print(f"ICD     → {EVAL_ICD_CSV}")
print(f"Per patient → {EXPORT_DIR}/patient_*/admissions/hadm_*/accuracy/")

In [ ]:
from pathlib import Path

sample = next(Path(EXPORT_DIR).glob("patient_*/admissions/hadm_*/accuracy/comparison.txt"), None)
if sample:
    print(sample)
    print(sample.read_text(encoding="utf-8")[:2500])
else:
    print("No comparison.txt yet — run the cell above.")